## Fixed-accuracy validation (2×3) — fig3-style replot

The files `*_holdout_points.csv` / `*_holdout_summary.csv` in `curve_fitting/plots_fixed_acc_validation_from_B1600/` are **walk-forward holdout diagnostics**: a few $(n_q,\, n_{ps})$ pairs per method/channel/$p$/target. They do **not** contain the full empirical curves or the dense extrapolation grid used for the main validation figure, so they cannot reproduce it by themselves.

This notebook rebuilds the validation figure using the cached bootstrap row JSONs from `curve_fitting/plots_unified_paper_full/ci68/` (see `curve_fitting/generate_plots.txt`), with `single_column.mplstyle` like `fig3_mf_protocols.ipynb`. The replot uses `plot_style="fig3_mf"`: $\epsilon_p \in \{0.01,0.05,0.1\}$ as Blues / Greens / Reds (lighter blue for the smallest $p$), $A_{\mathrm{target}}$ encoded by solid vs dashed model lines, empirical **markers only** (no connecting segments), the **model curve + CI on every** $n_q$ in the grid (including $n_q < 15$), and $n_q \le 30$ on the axis.

In [17]:
import importlib.util
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "figures_notebooks" else Path.cwd()

MPLSTYLE = REPO_ROOT / "figures_notebooks" / "single_column.mplstyle"
OUT_DIR = REPO_ROOT / "curve_fitting" / "plots_fixed_acc_validation_from_B1600"
BASE = "mf_fixedacc_validation_ch_dephasing_relaxation_p_0p01_0p05_0p1_targets_0p6_0p8_ci1"

ROWS_JSON = {
    "hypergraph": REPO_ROOT / "curve_fitting/plots_unified_paper_full/ci68/hg_bootstrap_threshold_rows.json",
    "eigenshadow": REPO_ROOT / "curve_fitting/plots_unified_paper_full/ci68/eigenshadow_bootstrap_threshold_rows.json",
    "ml": REPO_ROOT / "curve_fitting/plots_unified_paper_full/ci68/ml_bootstrap_threshold_rows.json",
}

channels = ["dephasing", "relaxation"]
amplitudes = [0.01, 0.05, 0.1]
targets = [0.6, 0.8]
nq_min, nq_max = 5, 30
ci_z = 1.0

In [18]:
spec = importlib.util.spec_from_file_location(
    "plot_mf_fixed_acc_validation",
    REPO_ROOT / "curve_fitting" / "plot_mf_fixed_acc_validation.py",
)
pfa = importlib.util.module_from_spec(spec)
sys.modules["plot_mf_fixed_acc_validation"] = pfa
spec.loader.exec_module(pfa)

unified = pfa._load_module("unified_fixed_acc_validation_mod", pfa.UNIFIED_PATH)

rows_by_case, predictors = {}, {}
for method in pfa.METHOD_ORDER:
    preloaded = unified._load_rows_json(ROWS_JSON[method])
    if method == "ml":
        for r in preloaded:
            r["channel"] = unified.ML_CHANNEL_CANONICAL.get(r.get("channel"), r.get("channel"))
        preloaded = unified._postprocess_rows(preloaded)
    for ch in channels:
        for amp in amplitudes:
            rows = pfa._filter_rows_case(preloaded, channel=str(ch), amplitude=float(amp))
            rows_by_case[(method, ch, float(amp))] = rows
            predictors[(method, ch, float(amp))] = pfa._build_predictor_for_method(unified, rows, method)

out_png = OUT_DIR / f"{BASE}_replot_fig3mf_nq30.png"
out_pdf = OUT_DIR / f"{BASE}_replot_fig3mf_nq30.pdf"

pfa.plot_validation(
    unified,
    predictors,
    rows_by_case,
    channels=channels,
    amplitudes=amplitudes,
    targets=targets,
    nq_min=nq_min,
    nq_max=nq_max,
    ci_z=ci_z,
    output_png=out_png,
    output_pdf=out_pdf,
    mplstyle_path=MPLSTYLE,
    plot_style="fig3_mf",
)
print("saved:", out_png)


Bad key axes.grid.alpha in file /Users/krzywdaja/Documents/noisy-learning-advantage/figures_notebooks/single_column.mplstyle, line 30 ('axes.grid.alpha   : 0.35')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.9.4/lib/matplotlib/mpl-data/matplotlibrc
or from the matplotlib source distribution


saved: /Users/krzywdaja/Documents/noisy-learning-advantage/curve_fitting/plots_fixed_acc_validation_from_B1600/mf_fixedacc_validation_ch_dephasing_relaxation_p_0p01_0p05_0p1_targets_0p6_0p8_ci1_replot_fig3mf_nq30.png


### Optional: what is in the holdout CSV

Scatter of observed vs predicted $\log_2 n_{ps}$ at held-out $n_q$ only (all `holdout_count` values pooled; duplicate keys averaged).

In [16]:
csv_path = OUT_DIR / f"{BASE}_holdout_points.csv"
df = pd.read_csv(csv_path)
gcols = ["method", "channel", "amplitude", "target", "nq_holdout"]
agg = (
    df.groupby(gcols, as_index=False)
    .agg(
        obs_log2_nps=("obs_log2_nps", "mean"),
        pred_log2_nps=("pred_log2_nps", "mean"),
        pred_status=("pred_status", lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0]),
    )
)

plt.style.use(str(MPLSTYLE))
method_order = ["hypergraph", "eigenshadow", "ml"]
method_labels = {
    "hypergraph": "Hypergraph",
    "eigenshadow": "Eigenshadow",
    "ml": "Shadow-based ML",
}
eps_c = pfa._epsilon_color_map_fig3(amplitudes)
tls = pfa._target_linestyle_map(targets)
fig, axes = plt.subplots(2, 3, figsize=(10.2, 5.6), sharex=True, sharey=True)
for i, ch in enumerate(channels):
    for j, meth in enumerate(method_order):
        ax = axes[i, j]
        sub = agg[(agg["channel"] == ch) & (agg["method"] == meth)]
        for amp in amplitudes:
            for tgt in targets:
                s = sub[np.isclose(sub["amplitude"], amp) & np.isclose(sub["target"], tgt)]
                if s.empty:
                    continue
                col = eps_c[float(amp)]
                ls = tls[float(tgt)]
                ax.plot(
                    s["nq_holdout"],
                    s["obs_log2_nps"],
                    linestyle="none",
                    marker="o",
                    color=col,
                    markerfacecolor=col,
                    markeredgecolor="black",
                    markeredgewidth=0.35,
                )
                ax.plot(
                    s["nq_holdout"],
                    s["pred_log2_nps"],
                    linestyle=ls,
                    marker="^",
                    color=col,
                    markerfacecolor="white",
                    markeredgewidth=0.8,
                    alpha=0.88,
                )
        if i == 0:
            ax.set_title(method_labels[meth])
        if j == 0:
            ax.set_ylabel(f"{ch.title()}\n" + r"$\log_2 n_{ps}$")
        if i == 1:
            ax.set_xlabel(r"$n_q$ (holdout)")
        ax.grid(True, linestyle="--", alpha=0.4)

fig.suptitle("Holdout CSV only: sparse $n_q$ (not the main validation figure)")
fig.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/krzywdaja/Documents/noisy-learning-advantage/curve_fitting/plots_fixed_acc_validation_from_B1600/mf_fixedacc_validation_ch_dephasing_relaxation_p_0p01_0p05_0p1_targets_0p6_0p8_ci1_holdout_points.csv'